# V7_0_N05 — From Signal to Accountable Action

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft. Illustrative data do not constitute official statistics or operational authorization.

## Learning outcomes
Transform analytical signals into governed decision products: scenarios, intervals, reason codes, dashboards, alerts, human review, monitoring, incident response, and retirement.

In [1]:
import numpy as np,pandas as pd,json,datetime
items=pd.DataFrame({'sector':['Agriculture','Health','Education','Habitat'],'signal':[.81,.74,.66,.58],'quality':[.93,.88,.91,.79],'capacity':[12,5,18,7],'owner':['Food security unit','Public health operations','District education authority','Urban planning authority']})
items

## 1. Separate signal from decision
A score becomes a review disposition only after quality, uncertainty, capacity, authority, and escalation rules are applied.

In [2]:
def route(r):
 if r.quality<.80:return 'ABSTAIN—QUALITY'
 if r.signal>=.70:return 'ESCALATE FOR REVIEW'
 if r.signal<.45:return 'ROUTINE MONITORING'
 return 'ABSTAIN—UNCERTAIN'
items['disposition']=items.apply(route,axis=1)
print(items.to_string(index=False))

     sector  signal  quality  capacity                        owner         disposition
Agriculture    0.81     0.93        12           Food security unit ESCALATE FOR REVIEW
     Health    0.74     0.88         5     Public health operations ESCALATE FOR REVIEW
  Education    0.66     0.91        18 District education authority   ABSTAIN—UNCERTAIN
    Habitat    0.58     0.79         7     Urban planning authority     ABSTAIN—QUALITY


## 2. Capacity-aware queues
An alert that cannot be reviewed is not actionable. Prioritization should expose unmet need rather than hide it.

In [3]:
items['estimated_cases']=[28,11,25,13]; items['queued']=items[['estimated_cases','capacity']].min(axis=1); items['unmet']=items.estimated_cases-items.queued
print(items[['sector','estimated_cases','capacity','queued','unmet']].to_string(index=False))

     sector  estimated_cases  capacity  queued  unmet
Agriculture               28        12      12     16
     Health               11         5       5      6
  Education               25        18      18      7
    Habitat               13         7       7      6


## 3. Executive dashboard contract
Every card needs a definition, reference period, geography, source vintage, uncertainty/status, owner, action, and link to evidence.

In [4]:
cards=[]
for r in items.itertuples():
 cards.append({'sector':r.sector,'indicator':'review signal','value':round(r.signal,2),'quality':round(r.quality,2),'status':r.disposition,'owner':r.owner,'unmet_capacity':int(r.unmet),'as_of':'2026-09-04'})
print(json.dumps(cards,indent=2))

[
  {
    "sector": "Agriculture",
    "indicator": "review signal",
    "value": 0.81,
    "quality": 0.93,
    "status": "ESCALATE FOR REVIEW",
    "owner": "Food security unit",
    "unmet_capacity": 16,
    "as_of": "2026-09-04"
  },
  {
    "sector": "Health",
    "indicator": "review signal",
    "value": 0.74,
    "quality": 0.88,
    "status": "ESCALATE FOR REVIEW",
    "owner": "Public health operations",
    "unmet_capacity": 6,
    "as_of": "2026-09-04"
  },
  {
    "sector": "Education",
    "indicator": "review signal",
    "value": 0.66,
    "quality": 0.91,
    "status": "ABSTAIN\u2014UNCERTAIN",
    "owner": "District education authority",
    "unmet_capacity": 7,
    "as_of": "2026-09-04"
  },
  {
    "sector": "Habitat",
    "indicator": "review signal",
    "value": 0.58,
    "quality": 0.79,
    "status": "ABSTAIN\u2014QUALITY",
    "owner": "Urban planning authority",
    "unmet_capacity": 6,
    "as_of": "2026-09-04"
  }
]


## 4. Alert deduplication and escalation
Repeated signals should update one case/event rather than create alert fatigue. Escalation is role-based and time-bound.

In [5]:
events=pd.DataFrame({'event_id':['E1','E1','E2'],'observed_at':['08:00','10:00','09:00'],'severity':[2,3,2]})
current=events.sort_values('observed_at').groupby('event_id',as_index=False).tail(1)
current['response_due_hours']=np.where(current.severity>=3,2,8)
print(current.to_string(index=False))

event_id observed_at  severity  response_due_hours
      E2       09:00         2                   8
      E1       10:00         3                   2


## 5. Monitoring after deployment
Monitor data quality, coverage/calibration where targets mature, workload, overrides, unresolved cases, incidents, and subgroup/geographic effects.

In [6]:
monitor=pd.DataFrame({'week':[1,2,3,4],'quality_pass':[.96,.94,.88,.76],'override_rate':[.12,.13,.19,.31],'unresolved_rate':[.08,.10,.18,.29]})
monitor['breach']=(monitor.quality_pass<.85)|(monitor.override_rate>.25)|(monitor.unresolved_rate>.25)
print(monitor.to_string(index=False))

 week  quality_pass  override_rate  unresolved_rate  breach
    1          0.96           0.12             0.08   False
    2          0.94           0.13             0.10   False
    3          0.88           0.19             0.18   False
    4          0.76           0.31             0.29    True


## 6. Incident and retirement controls
A material breach triggers containment, notification, investigation, correction, validation, and documented restart—or retirement.

In [7]:
latest=monitor.iloc[-1]; action='SUSPEND AND INVESTIGATE' if latest.breach else 'CONTINUE MONITORING'
incident={'trigger_week':int(latest.week),'action':action,'owner':'Service governance board','required_evidence':['data-quality diagnosis','affected decisions','containment log','revalidation','restart or retirement approval']}
print(json.dumps(incident,indent=2))

{
  "trigger_week": 4,
  "action": "SUSPEND AND INVESTIGATE",
  "owner": "Service governance board",
  "required_evidence": [
    "data-quality diagnosis",
    "affected decisions",
    "containment log",
    "revalidation",
    "restart or retirement approval"
  ]
}


## 7. Decision brief
Executives need the decision, evidence, uncertainty, affected groups, options, recommendation, owner, deadline, safeguards, and follow-up—not an unexplained model score.

In [8]:
brief={'decision':'Whether to continue the analytical service','evidence':'Quality and override thresholds breached in week 4','uncertainty':'Root cause not yet established','recommendation':'Suspend new alerts; preserve logs; investigate','authority':'Service governance board','next_review':'After documented revalidation'}
print(json.dumps(brief,indent=2))

{
  "decision": "Whether to continue the analytical service",
  "evidence": "Quality and override thresholds breached in week 4",
  "uncertainty": "Root cause not yet established",
  "recommendation": "Suspend new alerts; preserve logs; investigate",
  "authority": "Service governance board",
  "next_review": "After documented revalidation"
}


## Exercises
1. Add alert expiry and acknowledgement. 2. Define an override review. 3. Distinguish monitoring from impact evaluation. 4. Draft a retirement record.

## Exact solutions
1. Store created, acknowledged, resolved, and expiry times with escalation on missed service levels. 2. Review rates, reasons, outcomes, users, sectors/groups, and whether rules or training need change. 3. Monitoring checks operation and model behaviour; impact evaluation estimates whether the service caused better outcomes. 4. Record reason, authority, effective date, affected users, replacement, data/model disposition, retained evidence, and communications.

In [9]:
assert items.owner.notna().all() and monitor.breach.any()
assert incident['action']=='SUSPEND AND INVESTIGATE'
print('V7_0_N05_COMPLETE_EXECUTION_PASS')

V7_0_N05_COMPLETE_EXECUTION_PASS
